# Evaluate Performance Using CatBoost

In [2]:
# Import statements (add as needed)

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.compose import (
    TransformedTargetRegressor,
    make_column_transformer,
)

from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
from missforest import MissForest

from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from shap_select import shap_select
import xgboost as xgb
from xgboost import XGBRegressor
import optuna

/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# load both feature engineered dataset.

fe_df = pd.read_csv('../data/feature_engineered_training_set.csv')

In [4]:
fe_df = fe_df.drop(columns=['Unnamed: 0'])
fe_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,15845.0,11190.0,11426.0,...,0.000001,0.008799,0.488024,1800.0,1725.0,10077420.0,8.676030e+07,9.502319e+07,-37.216627,1.142857
1,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,15190.0,17658.5,9550.0,...,0.007830,0.458929,97.342756,1755.0,1365.0,116294640.0,3.369889e+05,2.440265e+05,-0.523901,0.931034
2,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,14879.0,15210.0,10720.0,...,0.006964,0.393498,96.911494,1472.0,1344.0,93380604.0,2.800000e+01,2.100000e+01,-0.575803,0.821429
3,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,15228.5,14887.0,10943.0,...,0.007228,0.409018,102.307634,1495.0,1430.0,60259174.5,4.884100e+05,4.132700e+05,-0.538889,0.884615
4,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,15130.0,16828.5,9502.5,...,0.004980,0.437815,96.513882,1600.0,1280.0,111901480.0,1.337840e+05,9.556000e+04,-0.784851,0.892857


In [5]:
train_df, test_df = train_test_split(fe_df, test_size=0.3)      # create train set and test set to eval performance

In [6]:
train_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
8733,10.000,33.00,20.0,194.2,1448.675974,1447.0,2692.0,15024.5,13349.0,9881.5,...,8.538365e-07,0.140081,0.000000,1281.0,1403.0,40445954.0,1.737231e+03,1.664846e+03,-35.556652,0.875000
7830,33.233,78.70,31.0,157.8,1471.364299,2906.0,5390.0,14712.5,15249.5,9208.5,...,1.060991e-03,0.374458,17.855424,1276.0,1450.0,79300375.0,2.145000e+03,1.625000e+03,-2.076924,0.666667
4539,61.612,283.95,73.5,153.8,267.299073,2443.0,4669.0,15118.0,13641.0,8893.0,...,1.993477e-04,0.327737,5.750098,1525.0,1647.0,70585942.0,1.701420e+05,1.531278e+05,-7.066055,0.833333
5196,65.914,1210.00,10.0,181.7,123.834243,2297.5,4231.0,15329.0,15875.0,10427.0,...,1.043807e-04,0.300412,51.294686,1608.0,1742.0,64856999.0,4.851064e+01,5.255319e+01,-30.923247,1.000000
764,183.205,463.00,10.0,163.1,1488.994033,1211.0,2724.5,14458.0,10501.5,9743.0,...,1.222149e-06,0.223982,0.000000,1550.0,1240.0,39390821.0,1.367932e+06,1.052255e+06,-196.326572,0.961538


In [7]:
train_df.shape

(6523, 33)

In [11]:
# Split train_df and test_df into X_train, X_test, y_train and y_test

X_train = train_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_train = train_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

X_test = test_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_test = test_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

### Preprocess Data - Catboost doesn't require Imputation or Scaling, can essentially skip this step.

## **Feature Selection** 

In [14]:
# Will need to do different feature selection again for CatBoost since preprocessing steps are different than SVR.
# Conduct feature selection using shap_select again.

def perform_feature_selection(X_train, y_train):
    results_dict = {}

    X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_train, test_size=0.2)
    X_te_df = pd.DataFrame(X_te, columns=X_train.columns, index=y_te.index)
    
    
    for target in y_train.columns:
        model = XGBRegressor(n_estimators=1000, verbosity = 0, eval_metric='rmse', objective='reg:squarederror')
        model.fit(X_tr, y_tr[target], eval_set=[(X_te, y_te[target])])

        selected_df = shap_select(model, X_te_df, y_te[target], task="regression", threshold=0.05)
        results_dict[target] = selected_df

    return results_dict  

In [ ]:
results = perform_feature_selection(X_train, y_train)

[0]	validation_0-rmse:58.85491
[1]	validation_0-rmse:49.90454
[2]	validation_0-rmse:43.98121
[3]	validation_0-rmse:39.98657
[4]	validation_0-rmse:37.09691
[5]	validation_0-rmse:35.05922
[6]	validation_0-rmse:34.03959
[7]	validation_0-rmse:33.39492
[8]	validation_0-rmse:32.66944
[9]	validation_0-rmse:31.96229
[10]	validation_0-rmse:31.51794
[11]	validation_0-rmse:31.27404
[12]	validation_0-rmse:31.06127
[13]	validation_0-rmse:30.93100
[14]	validation_0-rmse:30.66770
[15]	validation_0-rmse:30.61829
[16]	validation_0-rmse:30.24094
[17]	validation_0-rmse:30.12736
[18]	validation_0-rmse:29.86075
[19]	validation_0-rmse:29.73168
[20]	validation_0-rmse:29.58639
[21]	validation_0-rmse:29.61858
[22]	validation_0-rmse:29.61217
[23]	validation_0-rmse:29.52397
[24]	validation_0-rmse:29.41544
[25]	validation_0-rmse:29.36100
[26]	validation_0-rmse:29.31721
[27]	validation_0-rmse:29.21089
[28]	validation_0-rmse:29.19532
[29]	validation_0-rmse:29.12903
[30]	validation_0-rmse:28.97629
[31]	validation_0-